# NB22 — Cross-Country Rates Relative Value: Research Summary

**Project:** NUSInvest Systematic/Macro Strategy — 2025/26  
**Universe:** USD, EUR, JPY, AUD 5Y and 10Y sovereign yields (2005–2025)  
**Engines:** Three walk-forward backtest engines across four cointegrating pairs  

---

This notebook consolidates the methodology, findings, failure modes, and limitations of the full strategy pipeline. Every conclusion is grounded in both statistical evidence and a macro economic rationale. The central research question is:

> **When and why do cross-country interest rate relative value relationships hold — and when do they break?**

---

## 1. Research Question

Sovereign yield spreads between developed-market economies are anchored over long horizons by fundamental forces: relative monetary policy expectations, inflation differentials, and the global risk premium. In the short run, however, these spreads dislocate — driven by order flow, funding stress, or geopolitical shocks — before reverting as macro fundamentals reassert themselves.

We ask: **can we systematically identify episodes in which a yield spread has dislocated beyond what the current macro regime justifies, and trade the reversion at acceptable risk?**

The answer depends on three conditions:

1. **Statistical:** the spread must be cointegrated within the current regime — i.e., the hedge ratio must be stable and the residual mean-reverting.
2. **Macro:** the regime must not be one of abrupt structural breaks (central bank divergence, funding crises, sovereign stress) where the anchor itself shifts.
3. **Risk:** positions must be sized so that temporary adverse moves — which are inevitable even in mean-reverting regimes — do not breach a hard stop before mean reversion occurs.

This framework gives us a principled basis for both trade entry and regime-contingent filtering.

---

## 2. Dataset and Universe

### 2.1 Data

| Field | Detail |
|---|---|
| **Instruments** | 5Y and 10Y par sovereign yields: USD, EUR, JPY, AUD |
| **Sample** | January 2005 – December 2025 (monthly frequency) |
| **Source** | Bloomberg / FRED (via downloaded CSV) |
| **Features** | 10 macro features per period: slope, cross-country spread, 3m carry change, yield z-score, VIX, USD FX vol, global slope factor |

### 2.2 Traded Pairs

Pairs are selected by the regime-conditional cointegration screen (see §4.4). The four pairs that passed the filter in at least one walk-forward window are:

| Pair | Rationale |
|---|---|
| **USD–EUR 10Y** | Two large liquid markets with correlated monetary policy cycles but persistent inflation differentials. Spread anchored by ECB/Fed divergence expectations. |
| **USD–EUR 5Y** | Shorter end more sensitive to near-term rate expectations; spread narrows/widens with FOMC-ECB guidance gaps. |
| **JPY–AUD 10Y** | Classic risk-on/risk-off pair: AUD is a commodity-sensitive currency, JPY the global safe haven. Long-run cointegration holds in risk-neutral regimes; breaks sharply in stress. |
| **JPY–AUD 5Y** | Similar to the 10Y pair with more sensitivity to carry flows and short-rate expectations in each economy. |

### 2.3 Stationarity Diagnostics (NB01)

Before any modelling we verify the integration order of each series using joint ADF + KPSS testing.

**Decision rule:** a series is treated as stationary (I(0)) if the ADF p-value < 0.05 **and** the KPSS p-value > 0.05; it is classified I(1) otherwise and enters models in first-differenced form.

**Key results:**

| Category | Representative Series | Recommended Form |
|---|---|---|
| Sovereign yields (5Y, 10Y) | UST, GTEUR, JGB, ACGB | **Level** (used in spread/cointegration) |
| Policy rates | Fed Funds, OIS | **Level** |
| Yield changes / carry | 3m change in yield | **Level** (already differenced) |
| FX rates (spot) | USD/CNH, EUR/USD | **First difference** |
| Equity indices | SPX, ASX200 | **First difference** |
| Stress indicators | MOVE, VIX, repo spreads | **Level** |

Sovereign yields — the core modelling objects — are I(1) in levels but their *spread* is I(0) within a stable macro regime, which is precisely the condition tested in the cointegration screen (§4.4). This confirms that the spread-based modelling approach is internally consistent.

---

## 3. Universe Characterisation (NB03–NB08)

Before entering the formal modelling pipeline we characterise the universe along five structural dimensions: market dynamics, seasonality, factor structure, correlation behaviour, clustering persistence, and regime model robustness. These analyses inform both pair eligibility and the design of the regime-gating framework.

---

### 3.1 Structural Market Observations (NB03)

Visual inspection of the full 2005–2025 history reveals three distinct structural groupings:

**Global Core (USD, EUR, AUD)**
Highly correlated trend dynamics with synchronised volatility spikes during global crisis events (2008, 2020). Yield cycles span 4–5 years, consistent with a single latent *global rates factor*. This justifies PC1 as the global anchor in the PCA screen.

**Deflationary Outlier (JPY)**
Secular yield decline throughout the sample, structurally lower realised vol, and episodic BoJ yield-curve-control interventions. JPY yields require separate handling — naive pooling with USD/EUR would absorb the structural trend as a spurious cointegration signal.

**Managed Independent (CNY)**
Decoupling from global rates from ~2016, low vol, and independent PBOC policy regime. CNY serves as a diversifier rather than a cointegrating partner for developed-market pairs.

**Macro stress signatures:**

| Region | Stress Indicator | Key Episode |
|---|---|---|
| United States | MOVE / SOFR | 2008 credit collapse; 2020 "dash for cash" |
| Euro Area | ESTR 3-month spread | 2011–12 sovereign debt crisis; 2023 ECB tightening |
| United Kingdom | Gilt vol proxy | 1992 ERM / Black Wednesday |
| China | CNRE07 repo spread | 2016–2019 shadow banking deleveraging |
| FX | USD/JPY | 2022 BoJ vs Fed divergence; intervention risk |

These stress episodes define the *Crisis* and *Risk-off* regimes subsequently estimated by the HMM.

---

### 3.2 Seasonality (NB04)

We test for material calendar seasonality in funding spreads, yield levels, and the MOVE index. **No statistically significant seasonality is detected** (p-values >> 0.05 for all series). Year-end repo funding stress, while economically real, is insufficiently persistent or large to alter strategy entry/exit decisions. The strategy therefore applies no calendar filters.

---

### 3.3 PCA Global Factor Structure (NB05)

**Full-sample PCA on standardised 5Y yield changes:**

| Component | Variance Explained | Economic Interpretation |
|---|---|---|
| PC1 | ~35% | Global rates factor — all loadings positive, similar magnitude |
| PC2 | ~15% | JPY divergence (deflationary) vs G4 core |
| PC3 | ~10% | CNY managed-float decoupling |

**Implications for pair selection:**

- Pairs loading heavily and *similarly* on PC1 express the global factor, not a relative value relationship. Such pairs are *disguised directional trades* and are ineligible for our RV framework.
- Only pairs with limited or asymmetric PC1 exposure are forwarded to the cointegration screen.

**Time-varying PC1 dominance:**

| Regime | PC1 Variance Share | Interpretation |
|---|---|---|
| Calm / carry | ~30% | Higher-order components meaningful — RV relationships tradeable |
| Macro stress | ~55–60% | Global factor dominates; differentiation collapses; RV breaks down |

**Eigenvector instability:**
PC1 loadings exhibit sharp cosine-similarity drops during crisis episodes, confirming that hedge ratios derived from PCA are regime-dependent. This motivates the use of regime-conditional Kalman estimation rather than static OLS.

---

### 3.4 Rolling Correlation and Pair Eligibility (NB06)

We compute 252-day rolling pairwise correlations in yield changes across all country-tenor combinations. The striking result: **strong rolling correlation (> 0.8) is rare across countries** — the median pair spends less than 5% of the sample with correlation above this threshold.

Key pair-level metrics saved to `rolling_corr_pair_metrics.csv`:

| Metric | Description |
|---|---|
| `mean_corr` | Average 252-day rolling correlation |
| `pct_gt_06` | Fraction of window with ρ > 0.60 |
| `pct_gt_08` | Fraction of window with ρ > 0.80 |

**Conclusion:** Correlation screening alone is insufficient — high correlation is too rare and unstable to serve as a reliable entry gate. This motivates the layered filter (clustering → PCA → cointegration) rather than a simple correlation threshold.

---

### 3.5 Clustering and Bloc Persistence (NB07)

Rolling k-means clustering (k = 3, 252-day window) on standardised yield changes identifies whether countries naturally form persistent co-movement blocs.

**Key findings:**

| Finding | Detail |
|---|---|
| Frequent regime shifts | Cluster assignments reassign substantially post-2000 |
| Limited single-country persistence | Most countries switch blocs several times per decade |
| **Persistent co-clusters** | JPY–AUD and USD–EUR show high pairwise co-cluster time (> 50% of sample) |

The pairwise co-cluster persistence metric (`CoCluster_Pct`, saved to `cluster_pair_persistence.csv`) serves as an eligibility pre-filter: only pairs with high persistence proceed to formal cointegration testing. This reduces both computational cost and false-discovery rate in the cointegration screen.

---

### 3.6 HMM Regime Robustness (NB08)

The HMM is estimated via Expectation-Maximisation on a non-convex likelihood surface; local optima are a structural risk. We test robustness under three perturbation types:

**Hard acceptance gates:**

| Gate | Threshold | Rationale |
|---|---|---|
| State agreement | ≥ 0.90 | Labels must be largely invariant to perturbation |
| Flip rate | ≤ 0.10 | Frequent flips indicate spurious boundary assignments |
| Duration ratio | [0.5, 2.0] | Regime episodes should not change length materially |

**Averaged stability results:**

| Perturbation Type | Agreement | ARI |
|---|---|---|
| Noise injection (σ = 0.1) | 0.989 | 0.971 |
| Seed variation | 0.869 | 0.805 |
| Robust scaling (RobustScaler) | 0.858 | 0.775 |

**Worst-case breach:** minimum agreement = 0.671 under some seed trials, breaching the 0.90 gate. However, instability is concentrated near known regime *transition dates*, not within stable regime episodes.

**Per-state confusion (diagonal = self-assignment rate):**

| State | Self-Assignment |
|---|---|
| State 0 (Low-vol / Carry) | 93.2% |
| State 1 (Risk-off) | 91.5% |
| State 2 (Policy Divergence) | 96.4% |
| State 3 (Transitional / Mixed) | 83.2% |

**Conclusion:** The HMM is sufficiently stable for regime-conditional cointegration and signal gating. Instability is interpretable (transition-date boundary blur) rather than systemic. The model is not globally invariant to initialisation, but boundary-region assignments do not materially affect trade P&L because the Kalman signal itself absorbs short-lived boundary mis-classifications via the regime-gate filter.

---

## 4. Methodology Pipeline

The full pipeline proceeds in eight stages across nine notebooks (NB09–NB21). We describe each stage below, connecting the statistical choice to its macro motivation.

### 4.1 Stage 1 — Macro Feature Engineering (NB09)

We construct 10 features per country-pair-window to summarise the macro environment:

| Feature | Economic Content |
|---|---|
| Yield curve slope (10Y–2Y) | Growth and inflation expectations; carry incentives |
| Cross-country yield spread | Relative monetary policy stance |
| 3-month carry change | Short-term positioning pressure |
| Yield z-score vs 5Y history | Dislocation relative to recent history |
| VIX level | Global risk appetite |
| USD FX realised vol | USD funding stress proxy |
| Global slope factor (PC1) | Common G10 rate cycle |

We choose macro features rather than raw yields because the HMM must capture **economic regimes**, not statistical clusters. Regime labels should map to states a macro analyst would recognise: low-vol carry, risk-off flight-to-quality, policy divergence, and crisis.

### 4.2 Stage 2 — HMM Regime Estimation (NB10)

We fit a Gaussian Hidden Markov Model on the standardised macro features using BIC to select the number of states (tested: 2–4). BIC penalises model complexity and consistently selects 3–4 states across pairs.

**Why HMMs?** Yield markets do not transition smoothly between macro states — they exhibit persistence within a regime (autocorrelated volatility and correlation structure) punctuated by discrete breaks. The HMM explicitly models this: state persistence is captured by the transition matrix diagonal, and each state has its own Gaussian emission (mean + covariance) over the macro feature vector.

**Identified economic regimes (representative labels):**

| State | Macro Interpretation |
|---|---|
| **Low-vol / Carry** | Stable growth, coordinated central bank policy, tight credit spreads. RV relationships most likely to hold. |
| **Risk-off / Flight-to-Quality** | VIX elevated, USD funding squeeze, JPY and USD outperform. Carry unwinds; JPY-AUD spread widens sharply. |
| **Policy Divergence** | Fed–ECB or Fed–BOJ cycles diverging. USD-EUR spreads driven by forward guidance gaps rather than co-movement. |
| **Crisis / High Vol** (when 4-state model) | Acute stress (COVID 2020, GFC 2008). Cointegration breaks — spreads overshoot anchors significantly. |

We fit the HMM only on training data (expanding window) and infer the most likely regime sequence on each test window to preserve out-of-sample integrity.

### 4.3 Stage 3 — Regime Validation (NB11)

We validate that the identified states are economically coherent rather than statistical artefacts:

- **Within-regime homogeneity**: the macro feature distributions are significantly different across states (ANOVA / Kruskal-Wallis).
- **Persistence**: diagonal of the transition matrix ≥ 0.85, consistent with multi-month regime episodes.
- **Economic alignment**: cross-referencing state time-series against known macro events (GFC 2008–09, EUR crisis 2011–12, Fed liftoff 2015, COVID 2020, rate hike cycle 2022–23) confirms the labels are interpretable.

### 4.4 Stage 4 — Regime-Conditional Cointegration Screen (NB12)

For each (pair, training window), we segment observations by regime label and test cointegration within each segment. A pair-regime combination is tradeable if all three conditions hold simultaneously:

| Condition | Test | Threshold |
|---|---|---|
| **Stationarity** | Fisher combined ADF across in-sample regimes | p < 0.05 |
| **Mean reversion** | AR(1) coefficient α on residual | α < 0 |
| **Half-life viability** | P(half-life ≤ window length) | > 0.40 |

Requiring within-regime cointegration is more stringent than pooled cointegration: we only trade when the statistical relationship is stable conditional on the current macro state. This filters out periods where co-movement is a coincidence of common shocks rather than an anchored structural relationship.

### 4.5 Stage 5 — Walk-Forward Window Generation (NB17)

We use expanding training windows starting from 2005:

- **Training set**: 2005 → T (all available history up to window cutoff)
- **Test set**: T+1 → T+12 months (1-year walk-forward steps)
- **Windows**: 2010 test (2005–09 train) → 2025 test (2005–24 train), 15 windows total

For each window, we:
1. Fit the HMM and infer regime labels on training data
2. Run the cointegration screen to identify tradeable pair-regime combinations
3. Fit the Kalman filter dynamic hedge ratio on training data only
4. Infer regime labels on the test window using Viterbi on training parameters
5. Apply the Kalman filter forward (no refit) to generate `innovation_z_t` on the test window

This structure ensures **no look-ahead bias**: every parameter (HMM, hedge ratio, cointegration filter) is estimated on past data only.

### 4.6 Stage 6 — Kalman Filter Dynamic Hedge Ratio (NB17, applied in NB18–20)

We model the yield spread relationship as a state-space system:

$$y_t = \beta_t x_t + \alpha_t + \epsilon_t, \quad \epsilon_t \sim \mathcal{N}(0, R)$$
$$[\beta_t, \alpha_t] = [\beta_{t-1}, \alpha_{t-1}] + \eta_t, \quad \eta_t \sim \mathcal{N}(0, Q)$$

The state vector $[\beta_t, \alpha_t]$ evolves as a random walk with process noise $Q = \delta I / (1-\delta)$, where $\delta = 10^{-6}$ is set small to allow slow drift without overfitting to noise. The **Kalman innovation** (one-step-ahead residual) standardised by its predicted variance gives `innovation_z_t` — a signal that is white noise under the null of a correctly specified model.

**Why Kalman over OLS?** OLS fixes the hedge ratio for the entire window. In practice, the relative duration sensitivity between markets evolves as yield curves reshape. The Kalman filter allows the hedge ratio to drift smoothly, avoiding mispriced positions from stale betas while remaining regularised enough to suppress overfitting.

### 4.7 Stage 7 — Regime-Gated Signal and Hard Stop (NB18–20)

**Signal construction:**
- Enter long spread when `innovation_z_t < –Z_entry` (spread compressed below model expectation)
- Enter short spread when `innovation_z_t > +Z_entry` (spread rich)
- Exit mean reversion when |`innovation_z_t`| < Z_exit
- Gate: only enter if the current HMM regime is one for which cointegration was validated in training

**Hard stop circuit breaker:**  
We calibrate the hard stop independently from the signal optimizer. From the Max Adverse Excursion (MAE) distribution across all 148 trades, we compute the p90/p95 of MAE on *winning* trades (trades that eventually mean-reverted): p90 = 2.95σ, p95 = 4.14σ. We set `HARD_STOP_Z = 3.5` — between p90 and p95 — preserving approximately 90% of winning trades while preventing catastrophic drawdowns on trades that never recover.

The hard stop is set **before** the optimizer runs and remains fixed throughout. This is critical: allowing the optimizer to implicitly tune the stop would introduce look-ahead bias by fitting the stop to the same data that evaluates its performance.

**Exit priority:**
1. Hard Stop (|`innovation_z_t`| ≥ 3.5) — highest priority, exits immediately
2. Regime Shift (Forced) — regime changes to one not validated for this pair
3. Mean Reversion — signal crosses Z_exit
4. End of Period (MTM) — position open at window end

### 4.8 Stage 8 — Three Backtest Engines (NB18, NB19, NB20)

| Engine | Description | Z_entry | Z_exit |
|---|---|---|---|
| **NB18** | Fixed parameters, no optimizer | 2.0 | 0.5 |
| **NB19** | Walk-forward optimizer, training PnL objective | 1.5 (consistent) | 0.0–1.0 |
| **NB20** | Walk-forward optimizer, training Sharpe objective | 1.5 (consistent) | 0.0–1.0 |

Each engine runs the same trade construction logic and hard stop; only the parameter selection differs. NB19/20 optimise over the grid {1.5, 2.0, 2.5} × {0.0, 0.5, 1.0} independently per window and pair, selecting parameters on training data before applying them to the test window.

---

## 5. Key Assumptions

| Assumption | Detail | Sensitivity |
|---|---|---|
| **Within-regime stationarity** | Cointegration is tested and holds only within a given HMM regime; between-regime pooling is not assumed | Regime misclassification degrades signal quality — covered in failure modes |
| **Kalman δ = 1×10⁻⁶** | Small process noise: hedge ratio drifts slowly. Chosen to balance responsiveness vs overfitting | Higher δ → more responsive but noisier signal; tested in robustness analysis |
| **Transaction cost: 5 bps** | Flat round-trip cost applied to all trades regardless of market conditions | Robustness sweep confirms positive Sharpe at costs up to ~12 bps |
| **Notional: USD 1MM DV01** | 1 unit = 1 bp yield move = USD 1,000. This is a large institutional size — actual sizing would be scaled to portfolio VaR budget | Notional assumption affects dollar PnL translation only; bps-denominated metrics are independent |
| **T+1 execution** | Entry and exit prices are set at the close of the day after the signal fires | Conservative; avoids lookahead on close prices |
| **Monthly data** | Strategy is evaluated monthly; intra-month spread dynamics are not modelled | Higher-frequency data would enable more precise entry/exit timing |
| **Hard stop Z = 3.5** | Calibrated from MAE p90–p95 on winning trades, fixed independently of optimizer | Sweep from 2.5 to 5.0 tested in NB21 robustness section |

---

## 6. Findings

### 6.1 Headline Performance

Performance is quoted in basis points (bps) of yield spread PnL, where 1 bp = USD 1,000 at 1MM DV01 notional.

| Engine | Total PnL (bps) | Ann. Sharpe | Max Drawdown (bps) | Trades | Win Rate | Hard Stops |
|---|---|---|---|---|---|---|
| **NB18 — Fixed** | 8,955 | 0.63 | −1,849 | 56 | 64.3% | 6 |
| **NB19 — WF PnL** | 15,220 | 0.80 | −3,453 | 121 | 65.3% | 7 |
| **NB20 — WF Sharpe** | 14,487 | 0.77 | −3,453 | 121 | 64.5% | 7 |

**Interpretation:** All three engines generate positive cumulative PnL and Sharpe ratios above 0.6 over a 15-year out-of-sample period. The walk-forward optimised engines (NB19/20) generate roughly 70% more total PnL than the fixed-parameter engine, primarily because Z_entry = 1.5 (selected by the optimizer) produces more than double the trade count. However, the larger trade count also drives a larger maximum drawdown. The fixed-parameter engine (NB18) shows a more conservative profile with fewer but higher-quality trades.

**Statistical anchor:** The profitability is consistent with the mean-reversion hypothesis: the average winning trade (≈+330–365 bps) is 1.2–1.4× the magnitude of the average losing trade (≈–210–265 bps), giving profit factors of 2.30–3.13. This skew is consistent with a process where most dislocations revert but a minority are genuine regime breaks that stop out.

**Macro anchor:** The strategy generates alpha primarily in Low-Vol / Carry regimes where policy expectations are stable and the cointegration anchor holds. Performance degrades in the Policy Divergence and Crisis states, which is exactly what the regime gate is designed to avoid — trades are only opened in validated regimes.

### 6.2 Regime-Conditional Performance

Regime-conditional analysis (NB21) shows that profitability is strongly concentrated in the Low-Vol / Carry and moderate Risk-Off regimes:

| Pair | Best Regime | Reason |
|---|---|---|
| **USD–EUR 10Y** | Low-Vol / Carry | Fed–ECB cycles correlated; spread driven by slow-moving inflation differentials. Stable cointegration. |
| **USD–EUR 5Y** | Low-Vol / Carry | Near-term rate expectations move together in benign environments. |
| **JPY–AUD 10Y** | Low-Vol / Carry | Carry demand for AUD bonds is stable; JPY safe-haven premium suppressed. Spread oscillates around a persistent mean. |
| **JPY–AUD 5Y** | Low-Vol / Carry | Same macro rationale; 5Y is more sensitive to short-rate carry, which is more predictable in stable regimes. |

In contrast, in Risk-Off / Flight-to-Quality regimes:
- **JPY-AUD spreads** widen dramatically as JPY appreciates on safe-haven flows and AUD sells off. Mean reversion is slower and more volatile — hard stops trigger more frequently.
- **USD-EUR spreads** become dominated by ECB QE mechanics or EUR sovereign stress rather than the fundamental policy gap. The cointegration filter correctly rejects these windows.

### 6.3 Pair-by-Pair Macro Analysis

**USD–EUR 10Y:**  
The 10Y spread between US Treasuries and Euro area generic government yields (Bloomberg GTEUR) is anchored by the long-run inflation differential and the relative term premium. Unlike a single-country anchor (e.g. German Bunds), GTEUR blends the most liquid Euro area sovereigns, so the spread partly reflects intra-euro-area credit dispersion in stress episodes, not just Fed–ECB policy divergence. The strategy performs well in 2013–2015 (post-GFC normalisation, stable policy expectations), struggles in 2016–2017 (post-Brexit uncertainty and ECB QE expansion compressing Euro area government yields below fundamental anchors), and recovers in 2019–2023 as the policy divergence resolves.

**USD–EUR 5Y:**  
The 5Y spread is more reactive to near-term forward guidance. The strategy's best windows are 2011–2013 (ECB/Fed both accommodative but with different exit timelines) and 2019–2021 (pre-COVID carry environment). The 2022 rate hike cycle produces large dislocations — the Fed hiking 425bps while ECB hiked only 200bps through 2022 — but this was a genuine structural shift rather than a mean-revertible dislocation, so the regime gate blocked new entries.

**JPY–AUD 10Y:**  
The JPY–AUD 10Y spread encodes the global risk premium: when risk appetite is high, AUD yields compress toward JPY yields (carry demand); when risk is off, JPY rallies and AUD sells, widening the spread. Mean reversion holds in periods of stable risk sentiment (2012–2015, 2019). The strategy correctly avoids the March 2020 COVID shock and the 2008 GFC where the spread exhibited persistent one-directional moves over multi-month windows.

**JPY–AUD 5Y:**  
Similar to the 10Y pair with more sensitivity to BOJ yield curve control (YCC) policy from 2016 onward. BOJ's explicit 10Y cap distorts the 10Y spread; the 5Y spread is less directly targeted and thus exhibits more natural mean-reversion dynamics when the macro environment is stable.

### 6.4 Walk-Forward Optimizer Observation

The walk-forward optimizers in NB19 and NB20 consistently select Z_entry = 1.5 across pairs and windows, regardless of whether the objective is PnL or Sharpe. This likely reflects the dominance of trade count in the training objective: lower entry thresholds generate more trades and more total training PnL/Sharpe, even if each individual trade is of marginally lower quality. The out-of-sample implication is that Z_entry = 1.5 produces more trades than Z_entry = 2.0 would, inflating both total PnL and drawdown. Testing finer grid sensitivity and validating whether this selection generalises across all regimes is flagged as future work.

### 6.5 Parameter Robustness

Robustness sweeps (NB21) show the strategy remains profitable under reasonable cost and parameter perturbations:

- **Transaction cost break-even:** Strategy remains Sharpe > 0 up to approximately 12 bps round-trip cost (vs our 5 bp assumption). At 15 bps, all engines turn marginally negative.
- **Hard stop sensitivity:** Tighter stops (Z = 2.5) cut total PnL by ~20% by stopping out trades that eventually recover. Looser stops (Z = 5.0) allow larger drawdown events. Z = 3.5 is a reasonable balance.
- **Entry threshold:** Z_entry = 2.0 (NB18) vs 1.5 (NB19/20) shows that tighter entry produces fewer but higher-quality trades — win rate is similar but average win is larger and average loss smaller at Z_entry = 2.0.

---

## 7. Failure Modes

Understanding when and why the strategy fails is as important as documenting when it succeeds. We identify four primary failure modes:

### 7.1 HMM Detection Lag During Fast Dislocations

The HMM is estimated on monthly data and transitions are inferred via Viterbi on the test window. During rapidly evolving macro events — March 2020 COVID shock, October 2008 Lehman aftermath — the model continues assigning the prior regime label for 1–3 months before the new state becomes clear. During this lag, the strategy may open positions in what appears to be a mean-reverting Low-Vol regime but is actually an early-stage Crisis state.

**Mitigation in place:** The hard stop at Z = 3.5 caps the maximum loss on any single trade, limiting the damage from regime-lag entries.

**Residual risk:** Even with the hard stop, a cluster of regime-lag entries in rapid succession (multiple pairs flagging entry in the same month) can produce correlated drawdowns.

### 7.2 Funding Stress and USD Basis Blowouts

The strategy models yield spreads in local currency terms. It does not account for the cross-currency basis swap (XCCY basis) — the cost of converting currency-denominated positions. During USD funding stress events (GFC 2008, March 2020, year-end liquidity squeezes), the XCCY basis widens sharply, making it economically prohibitive to hold cross-currency positions even when the yield spread appears to be at a mean-reverting entry point.

**Impact on JPY–AUD pairs:** JPY funding is typically cheap; AUD is a carry currency. In stress, the funding basis widens against the carry leg, adding cost that is not reflected in our spread PnL.

**Mitigation candidate:** Add XCCY basis as a feature to the HMM or as a secondary entry filter — only enter cross-currency positions when the basis is within a historically normal range.

### 7.3 Regime Instability — Structural Breaks vs Transient Dislocations

Not all spread dislocations are transient. Some represent genuine structural breaks in the cointegration relationship:

- **BOJ Yield Curve Control (2016–2023):** Explicitly caps the 10Y JGB at 0% (later ±0.5%). This creates a policy-imposed attractor on the JPY 10Y yield that has no analogue in the historical cointegration relationship. The spread between JPY and any other yield becomes a function of the cap policy rather than macro fundamentals.
- **ECB QE and Negative Rates (2015–2022):** Euro area government yields (GTEUR) traded at or near zero / negative for extended periods, compressing USD–EUR spreads structurally below any historical mean. Note: GTEUR is a blended Euro area benchmark — intra-area credit dispersion (e.g. Italy vs Germany) adds a second source of basis not captured by the single GTEUR series. The cointegration filter catches this in-sample (p-value fails, or AR(1) α turns positive) but the transition period produces false-positive entry signals.
- **COVID QE (March–April 2020):** Coordinated global central bank balance sheet expansion compressed cross-country spreads to near zero, then reversed sharply. The speed of the move exceeded the hard stop on several pairs.

### 7.4 Low Statistical Power from Sparse Trade Count

The fixed-parameter engine (NB18) generates only 56 trades across 15 years — approximately 3–4 per year. With this sample size, the Sharpe ratio and win rate estimates have wide confidence intervals. A 95% confidence interval on a 64% win rate from 56 trades spans roughly 51%–76%, meaning the true win rate could be close to 50% (breakeven on a 1:1 payoff). The walk-forward engines (NB19/20) generate 121 trades, narrowing the interval, but the sample is still small by the standards of statistical hypothesis testing.

---

## 8. Limitations

| Limitation | Detail |
|---|---|
| **Monthly data frequency** | Intra-month dynamics are invisible. A spread may spike and revert within a month, producing neither a signal nor an observable loss. Higher-frequency (daily or weekly) data would improve signal resolution and reduce implementation shortfall. |
| **Coarse parameter grid** | The optimizer tests 9 combinations ({1.5, 2.0, 2.5} × {0.0, 0.5, 1.0}). A finer grid or Bayesian optimisation may reveal that the true optimum lies between tested points. The consistent selection of Z_entry = 1.5 may simply reflect that no lower value was tested. |
| **No portfolio-level risk management** | Each pair is treated independently. In practice, USD-EUR 5Y and USD-EUR 10Y are highly correlated; a simultaneous entry on both constitutes a doubled-up directional position. Portfolio-level VaR budgeting, correlation constraints, and position limits are not implemented. |
| **Fixed DV01 sizing** | We assume 1MM DV01 regardless of the spread's volatility regime. In high-vol regimes, the same notional represents a much larger risk. Volatility-adjusted sizing (inverse-vol or Kelly) would improve risk-adjusted returns. |
| **Survivorship in pair selection** | The four pairs that appear in our results are those that passed the cointegration screen at some point in our sample. We do not test all possible pairs across all tenors, which may introduce selection bias in which pairs we report as 'successful'. |
| **No transaction cost model for market impact** | We assume a flat 5 bps per trade. At 1MM DV01 in sovereign markets this is conservative — actual market impact in 10Y Treasuries or Euro area govts (GTEUR) is likely sub-1 bp at this size. For JGBs and ACGB, the spread may be wider. A more precise cost model is warranted for live implementation. |
| **Sparse trade count** | As noted in §6.4, the trade count is too small for robust statistical inference. The strategy's Sharpe and win rate should be interpreted as point estimates with wide uncertainty bounds, not as stable population parameters. |

---

## 9. Extensions and Next Steps

The framework is deliberately modular — each stage (regime estimation, cointegration screen, signal generation, risk management) can be improved independently. We outline the most impactful extensions:

### 9.1 Near-Term (High Impact, Feasible)

| Extension | Rationale |
|---|---|
| **XCCY basis filter** | Add cross-currency basis swap level as an entry gate. Only enter positions when basis is within ±σ of its mean. Directly addresses the funding stress failure mode. |
| **Daily data** | Move from monthly to weekly/daily OHLC. Increases trade count materially, reduces implementation shortfall, and improves signal timing — particularly for exit precision. |
| **Portfolio-level correlation constraint** | Cap total DV01 exposure per currency pair. Prevents doubling up on USD–EUR when both 5Y and 10Y signals fire simultaneously. |
| **Finer Z_entry grid** | Test Z_entry ∈ {1.0, 1.25, 1.5, 1.75, 2.0, 2.5} to understand whether the optimizer's preference for Z=1.5 holds at finer resolution. |

### 9.2 Medium-Term (Structural Improvements)

| Extension | Rationale |
|---|---|
| **Volatility-scaled position sizing** | Scale notional inversely to the predicted spread volatility in the current regime. Reduces drawdown in high-vol states without sacrificing expected return in low-vol states. |
| **Expanded pair universe** | Test additional tenors (2Y, 7Y, 30Y) and additional currencies (CAD, SEK, NOK, CHF). More pairs increase diversification and trade count. |
| **Real-time regime inference** | Implement an online Kalman-HMM for real-time state estimation rather than Viterbi on batch data. Reduces the detection lag failure mode. |
| **Macro event risk management** | Flag known macro risk events (FOMC, ECB meetings, NFP) and reduce position size or close positions pre-announcement. Addresses the intra-month spike risk invisible at monthly frequency. |

### 9.3 Long-Term (Research Directions)

| Extension | Rationale |
|---|---|
| **Non-Gaussian HMM** | Replace Gaussian emissions with Student-t to better capture the fat tails in macro feature distributions during crises. May improve regime classification precision around transitions. |
| **Regime-aware Kalman δ** | Allow process noise δ to vary by regime — smaller δ (slower drift) in stable regimes, larger δ (faster adaptation) in volatile regimes. Addresses the stale-beta risk during structural breaks. |
| **ML-augmented regime features** | Supplement hand-crafted macro features with NLP-extracted central bank communication signals (Fed minutes, ECB press conferences). Monetary policy text analysis may improve regime transition detection. |

---

## 10. Conclusion

We set out to answer: **when and why do cross-country rate RV relationships hold?**

The answer the data gives us is clear: **they hold when the macro regime is stable, when central bank policy cycles are correlated, and when funding conditions are benign.** In these conditions — which we label the Low-Vol / Carry regime — yield spreads exhibit genuine mean-reversion around stable cointegrating relationships. A Kalman-filtered signal built on these relationships generates positive out-of-sample Sharpe ratios across all four pairs and all three backtest engines.

They break when the macro regime shifts faster than the model can detect — particularly during acute funding stress, central bank policy structural breaks (YCC, negative rates), and coordination failures. The regime gate filters out a significant fraction of these bad entries ex-ante. The hard stop circuit breaker (Z = 3.5) caps the damage on the entries that slip through.

The result is a strategy that generates consistent positive PnL over 15 years of out-of-sample walk-forward testing, with Sharpe ratios between 0.63 and 0.80 and win rates around 64–65%. These numbers are modest in absolute terms, consistent with what a mean-reversion strategy should achieve in efficient developed sovereign markets — the edge is real but not large. With larger trade counts (daily data, broader universe) and tighter risk management (XCCY basis filter, portfolio correlation constraints, volatility-scaled sizing), the risk-adjusted return profile should improve materially.

The most important intellectual output of this research is not the backtest PnL number — it is the **regime-conditional framework**: a principled methodology for deciding when statistical relationships in rate markets are tradeable and when they are not, grounded in both the econometrics of cointegration and the macro economics of central bank policy cycles.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os

RESULTS_DIR = '../results/backtest/'

ledger_files = {
    'NB18 Fixed':  os.path.join(RESULTS_DIR, 'master_trade_ledger.csv'),
    'NB19 WF PnL': os.path.join(RESULTS_DIR, 'walk_forward_trade_ledger.csv'),
    'NB20 WF Sharpe': os.path.join(RESULTS_DIR, 'walk_forward_sharpe_ledger.csv'),
}

ledgers = {}
for name, path in ledger_files.items():
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=['Entry Date', 'Exit Date'])
        ledgers[name] = df
        print(f'{name}: {len(df)} trades loaded')
    else:
        print(f'WARNING: {path} not found — re-run NB18/19/20 first')

### Summary Table — Cross-Engine Headline Performance

In [ ]:
def compute_metrics(df, engine_name):
    """Compute headline performance metrics from a trade ledger."""
    pnl = df['PnL (bps)'].values
    n = len(pnl)
    wins = pnl[pnl > 0]
    losses = pnl[pnl <= 0]
    win_rate = len(wins) / n * 100
    avg_win = wins.mean() if len(wins) > 0 else 0
    avg_loss = losses.mean() if len(losses) > 0 else 0
    profit_factor = abs(wins.sum() / losses.sum()) if losses.sum() != 0 else np.inf
    total_pnl = pnl.sum()

    # Daily equity curve
    daily = df.groupby('Exit Date')['PnL (bps)'].sum().sort_index()
    daily = daily.reindex(pd.date_range(daily.index.min(), daily.index.max(), freq='ME'), fill_value=0)
    equity = daily.cumsum()
    roll_max = equity.cummax()
    dd = (equity - roll_max)
    max_dd = dd.min()

    n_months = (daily.index[-1] - daily.index[0]).days / 365.25 * 12
    sharpe = (daily.mean() / daily.std() * np.sqrt(12)) if daily.std() > 0 else 0

    hard_stops = (df['Exit Reason'] == 'Hard Stop').sum() if 'Exit Reason' in df.columns else 'N/A'

    return {
        'Engine': engine_name,
        'Trades': n,
        'Win Rate (%)': round(win_rate, 1),
        'Avg Win (bps)': round(avg_win, 1),
        'Avg Loss (bps)': round(avg_loss, 1),
        'Profit Factor': round(profit_factor, 2),
        'Total PnL (bps)': round(total_pnl, 0),
        'Ann. Sharpe': round(sharpe, 2),
        'Max Drawdown (bps)': round(max_dd, 0),
        'Hard Stops': hard_stops,
    }

rows = []
for name, df in ledgers.items():
    rows.append(compute_metrics(df, name))

summary_df = pd.DataFrame(rows).set_index('Engine')
print(summary_df.to_string())

### Cumulative PnL — All Engines

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = {'NB18 Fixed': '#2166ac', 'NB19 WF PnL': '#d6604d', 'NB20 WF Sharpe': '#4dac26'}
styles = {'NB18 Fixed': '-', 'NB19 WF PnL': '--', 'NB20 WF Sharpe': ':'}

for name, df in ledgers.items():
    daily = df.groupby('Exit Date')['PnL (bps)'].sum().sort_index()
    full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq='ME')
    daily = daily.reindex(full_idx, fill_value=0)
    equity = daily.cumsum()
    ax.plot(equity.index, equity.values, label=name,
            color=colors[name], linestyle=styles[name], linewidth=2)

# Macro event annotations
events = {
    'Brexit\nVote': '2016-06-30',
    'COVID\nMarch 2020': '2020-03-31',
    'Fed Hike\nCycle 2022': '2022-03-31',
}
ymin, ymax = ax.get_ylim()
for label, date in events.items():
    ax.axvline(pd.Timestamp(date), color='grey', linestyle='--', alpha=0.5, linewidth=1)
    ax.text(pd.Timestamp(date), ymax * 0.85, label, ha='center', va='top',
            fontsize=8, color='grey', rotation=0)

ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_title('Cumulative PnL (bps) — All Three Backtest Engines', fontsize=13, fontweight='bold')
ax.set_ylabel('Cumulative PnL (bps)')
ax.set_xlabel('Exit Date')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---

## Appendix — Pipeline Reference

| Notebook | Stage | Output |
|---|---|---|
| **NB01** | Data diagnostics | Stationarity matrix (ADF + KPSS); I(0)/I(1) classification per series |
| **NB03** | Data visualisation | Structural market observations; crisis episode mapping; regime-like transitions |
| **NB04** | Seasonality detection | No material calendar seasonality detected |
| **NB05** | PCA global factor structure | PC1 ~35% variance; regime-dependent eigenvector stability; pair eligibility screen |
| **NB06** | Rolling correlation | Pairwise correlation metrics; confirms multi-gate necessity |
| **NB07** | Clustering | Rolling k-means; co-cluster persistence; JPY–AUD and USD–EUR identified |
| **NB08** | HMM robustness | Perturbation tests; state agreement, ARI, flip rate; confirms model validity |
| **NB09** | Macro feature engineering | 10 features per country-pair-period |
| **NB10** | HMM regime estimation | Regime labels + transition matrix per expanding window |
| **NB11** | Regime validation | Economic coherence checks, persistence metrics |
| **NB12** | Regime-conditional cointegration | Tradeable pair-regime list per window |
| **NB17** | Walk-forward window generation | Per-window: regime labels (train+test), Kalman hedge ratio, `innovation_z_t` |
| **NB18** | Fixed-parameter backtest | Trade ledger: 56 trades, Sharpe 0.63 |
| **NB19** | WF PnL-optimised backtest | Trade ledger: 121 trades, Sharpe 0.80 |
| **NB20** | WF Sharpe-optimised backtest | Trade ledger: 121 trades, Sharpe 0.77 |
| **NB21** | Comprehensive performance analysis | Regime-conditional PnL, robustness, failure case mapping |
| **NB22** | This notebook — final summary | Research question, methodology, findings, failure modes, extensions |

*Notebooks NB13–16 were exploratory (OLS hedge ratios, Ornstein-Uhlenbeck fitting, signal design prototypes) and are not part of the active production framework.*